# ML-07 — Refresh / Content Opportunity Scoring

This notebook checks two March-only signals against the April decline outcome, encodes one transparent review rule, and exports a ranked refresh queue. The April outcome is used only for retrospective signal checks — never in the score or exported queue.

## 1. Check Two Signals

**Decision moment:** March 31, 2026. The safe signals use March only: `ctr_gap_pp` is the item's CTR minus the March median CTR for its position band, in percentage points; `log1p_impressions_mar` is March impressions on a log scale. The outcome is `target_decline = 1` when April 1–30 impressions are below March 2–31 impressions.

The SQL reads only the exact March and April `data_0.parquet` files. Signal tables retain items with complete 30-day baseline and outcome windows and a positive baseline. CTR-gap buckets move from the most negative quarter of below-median March gaps to positive relative CTR; volume buckets use distinct cut points from March feature-population quartiles (repeated boundaries may reduce the number of buckets). Each table reports `n` and the observed target-decline mean.

Verdicts use a 1 percentage-point minimum spread: **CONFIRMED** means bucket rates follow the stated directional hypothesis, **OPPOSITE** means they follow the reverse direction, **MIXED** means the rates are non-monotonic, and **FALSE** means there is no material spread (or too few populated buckets). The CTR-gap hypothesis is that decline rates fall from the severe-negative to positive buckets. The volume hypothesis is that decline rates rise from low to high volume; volume is still treated as impact for prioritization, not as a causal risk factor.

In [1]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Set HF_TOKEN in the environment or a local .env file before running this notebook."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
escaped_hf_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE HUGGINGFACE, TOKEN '{escaped_hf_token}');"
)

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)
APRIL_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

features_and_target_sql = f"""
WITH march_daily AS (
    SELECT report_date, content_hash_id,
           CAST(gsc_impressions AS DOUBLE) AS gsc_impressions,
           CAST(gsc_clicks AS DOUBLE) AS gsc_clicks,
           CAST(gsc_avg_position AS DOUBLE) AS gsc_avg_position
    FROM read_parquet('{MARCH_PATH}', hive_partitioning = true)
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),
april_daily AS (
    SELECT report_date, content_hash_id,
           CAST(gsc_impressions AS DOUBLE) AS gsc_impressions
    FROM read_parquet('{APRIL_PATH}', hive_partitioning = true)
    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'
),
march_item_metrics AS (
    SELECT content_hash_id,
           COALESCE(SUM(gsc_impressions), 0.0) AS impressions_mar,
           COALESCE(SUM(gsc_clicks), 0.0) AS clicks_mar,
           SUM(CASE WHEN gsc_avg_position > 0
                    THEN gsc_avg_position * gsc_impressions END)
             / NULLIF(SUM(CASE WHEN gsc_avg_position > 0
                               THEN gsc_impressions END), 0) AS avg_position_mar
    FROM march_daily
    GROUP BY content_hash_id
),
march_item_rates AS (
    SELECT *,
           100.0 * clicks_mar / NULLIF(impressions_mar, 0) AS ctr_pct,
           CASE
               WHEN avg_position_mar > 0 AND avg_position_mar <= 3 THEN 'top_3'
               WHEN avg_position_mar > 3 AND avg_position_mar <= 10 THEN 'page_1'
               WHEN avg_position_mar > 10 AND avg_position_mar <= 20 THEN 'striking'
               WHEN avg_position_mar > 20 AND avg_position_mar <= 50 THEN 'page_3_5'
               WHEN avg_position_mar > 50 THEN 'deep'
               ELSE NULL
           END AS position_band
    FROM march_item_metrics
),
position_benchmarks AS (
    SELECT position_band, MEDIAN(ctr_pct) AS median_ctr_pct
    FROM march_item_rates
    WHERE position_band IS NOT NULL AND ctr_pct IS NOT NULL
    GROUP BY position_band
),
march_features AS (
    SELECT r.content_hash_id,
           LN(1 + r.impressions_mar) AS log1p_impressions_mar,
           r.ctr_pct - b.median_ctr_pct AS ctr_gap_pp
    FROM march_item_rates AS r
    LEFT JOIN position_benchmarks AS b USING (position_band)
),
baseline_window AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS baseline_impressions,
           COUNT(DISTINCT report_date) AS baseline_days
    FROM march_daily
    WHERE report_date >= DATE '2026-03-02'
      AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id
),
april_window AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS april_impressions,
           COUNT(DISTINCT report_date) AS april_days
    FROM april_daily
    GROUP BY content_hash_id
)
SELECT f.content_hash_id,
       f.log1p_impressions_mar,
       f.ctr_gap_pp,
       b.baseline_impressions,
       b.baseline_days,
       a.april_impressions,
       a.april_days,
       CASE WHEN a.april_impressions < b.baseline_impressions
            THEN 1 ELSE 0 END AS target_decline
FROM march_features AS f
LEFT JOIN baseline_window AS b USING (content_hash_id)
LEFT JOIN april_window AS a USING (content_hash_id)
"""

feature_frame = con.execute(features_and_target_sql).df()
if feature_frame.empty:
    raise RuntimeError("The March feature query returned no content items.")

# Labels are used only for retrospective signal checks, never for scoring.
labelled = feature_frame.loc[
    feature_frame["baseline_days"].eq(30)
    & feature_frame["april_days"].eq(30)
    & feature_frame["baseline_impressions"].gt(0)
].copy()
if labelled.empty:
    raise RuntimeError("No items have complete 30-day windows and a positive baseline.")
labelled["target_decline"] = labelled["target_decline"].astype("int8")

scoring_population = feature_frame.dropna(
    subset=["log1p_impressions_mar", "ctr_gap_pp"]
).copy()
negative_gap_values = scoring_population.loc[
    scoring_population["ctr_gap_pp"] < 0, "ctr_gap_pp"
]
if negative_gap_values.empty:
    raise RuntimeError("No March items have a negative CTR gap for a refresh rule.")
SEVERE_CTR_GAP_THRESHOLD_PP = float(negative_gap_values.quantile(0.25))

session_edges = [-np.inf, SEVERE_CTR_GAP_THRESHOLD_PP, 0.0, 2.0, np.inf]
session_labels = [
    f"Severe negative (<= {SEVERE_CTR_GAP_THRESHOLD_PP:.2f} pp)",
    f"Negative ({SEVERE_CTR_GAP_THRESHOLD_PP:.2f}, 0 pp]",
    "Near benchmark (0, 2 pp]",
    "Positive (> 2 pp)",
]
session_data = labelled.dropna(subset=["ctr_gap_pp"]).copy()
session_data["bucket"] = pd.cut(
    session_data["ctr_gap_pp"],
    bins=session_edges,
    labels=session_labels,
    right=True,
    ordered=True,
)
session_bucket_table = (
    session_data.groupby("bucket", observed=False)
    .agg(n=("target_decline", "size"), target_decline_mean=("target_decline", "mean"))
    .reset_index()
)

# Define volume buckets and the high-volume threshold from March features only.
volume_values = feature_frame["log1p_impressions_mar"].dropna()
if volume_values.empty:
    raise RuntimeError("No March impression volumes are available for bucketing.")
volume_edges = np.unique(volume_values.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy())
overall_volume_threshold = float(volume_values.quantile(0.75))
if len(volume_edges) == 1:
    volume_labels = ["All March volumes"]
    feature_frame["volume_bucket"] = pd.Categorical(
        ["All March volumes"] * len(feature_frame),
        categories=volume_labels,
        ordered=True,
    )
else:
    volume_edges[0] = -np.inf
    volume_edges[-1] = np.inf
    volume_labels = [f"Q{i} of March volume" for i in range(1, len(volume_edges))]
    feature_frame["volume_bucket"] = pd.cut(
        feature_frame["log1p_impressions_mar"],
        bins=volume_edges,
        labels=volume_labels,
        include_lowest=True,
        ordered=True,
    )

volume_data = labelled.dropna(subset=["log1p_impressions_mar"]).copy()
volume_data["bucket"] = feature_frame.loc[volume_data.index, "volume_bucket"]
volume_bucket_table = (
    volume_data.groupby("bucket", observed=False)
    .agg(n=("target_decline", "size"), target_decline_mean=("target_decline", "mean"))
    .reset_index()
)

def directional_verdict(bucket_table, expected_direction):
    populated = bucket_table.loc[bucket_table["n"] > 0, "target_decline_mean"]
    rates = populated.astype(float).to_numpy()
    if len(rates) < 2 or np.ptp(rates) < 0.01:
        return "FALSE", "The populated bucket rates span less than 1 percentage point."

    changes = np.diff(rates)
    aligned_changes = changes * expected_direction
    if np.all(aligned_changes >= -1e-12):
        return "CONFIRMED", (
            f"The ordered bucket rates change in the hypothesized direction "
            f"({rates[0]:.1%} to {rates[-1]:.1%})."
        )
    if np.all(aligned_changes <= 1e-12):
        return "OPPOSITE", (
            f"The ordered bucket rates move opposite to the hypothesis "
            f"({rates[0]:.1%} to {rates[-1]:.1%})."
        )
    return "MIXED", "The populated bucket rates are non-monotonic."

print("CTR-gap (Session Flag) buckets:")
display(session_bucket_table)
session_verdict, session_explanation = directional_verdict(
    session_bucket_table, expected_direction=-1
)
print(f"Session Flag verdict: {session_verdict} — {session_explanation}")

print("March impression-volume buckets:")
display(volume_bucket_table)
volume_verdict, volume_explanation = directional_verdict(
    volume_bucket_table, expected_direction=1
)
print(f"Volume verdict: {volume_verdict} — {volume_explanation}")
print(
    f"Whole-cohort March volume 75th percentile: "
    f"log1p_impressions_mar >= {overall_volume_threshold:.4f}"
)
print(
    f"Severe-negative CTR-gap cutoff (lower quartile among negative March gaps): "
    f"ctr_gap_pp <= {SEVERE_CTR_GAP_THRESHOLD_PP:.4f} pp"
)

CTR-gap (Session Flag) buckets:


,bucket,n,target_decline_mean
0,Severe negative (<= -0.07 pp),4360,0.704128
1,"Negative (-0.07, 0 pp]",85264,0.645079
2,"Near benchmark (0, 2 pp]",59404,0.630799
3,Positive (> 2 pp),2874,0.557759


Session Flag verdict: CONFIRMED — The ordered bucket rates change in the hypothesized direction (70.4% to 55.8%).
March impression-volume buckets:


,bucket,n,target_decline_mean
0,Q1 of March volume,8751,0.571135
1,Q2 of March volume,66284,0.617404
2,Q3 of March volume,77571,0.664269


Volume verdict: CONFIRMED — The ordered bucket rates change in the hypothesized direction (57.1% to 66.4%).
Whole-cohort March volume 75th percentile: log1p_impressions_mar >= 5.3799
Severe-negative CTR-gap cutoff (lower quartile among negative March gaps): ctr_gap_pp <= -0.0749 pp


## 2. Encode the Rule & Export

**One rule:** mark as severe underperformance the lowest quartile of negative March `ctr_gap_pp` values, then prioritize the highest-volume quartile within that severe-negative group. This relative definition avoids calling ordinary negative noise “severe” and uses no April outcome. The numeric score is March `log1p_impressions_mar`, so more visible qualifying items rank first. Each selected item receives the single reason code `High-Visibility_Underperforming-CTR` and the action `Review for Refresh`.

Scoring starts from March features only. The April target, April impressions, and baseline/target completeness are not inputs to the score or exported queue. The CSV contains only selected actions and safe context fields.

In [2]:
scoring_features = feature_frame[
    ["content_hash_id", "log1p_impressions_mar", "ctr_gap_pp"]
].copy()
assert "target_decline" not in scoring_features.columns
assert "april_impressions" not in scoring_features.columns

scoring_features = scoring_features.dropna(
    subset=["log1p_impressions_mar", "ctr_gap_pp"]
).copy()
has_severe_negative_gap = scoring_features["ctr_gap_pp"] <= SEVERE_CTR_GAP_THRESHOLD_PP
severe_negative_items = scoring_features.loc[has_severe_negative_gap]
if severe_negative_items.empty:
    raise RuntimeError("The March-only severity cutoff selected no items.")
high_volume_threshold = float(
    severe_negative_items["log1p_impressions_mar"].quantile(0.75)
)
is_high_volume = scoring_features["log1p_impressions_mar"] >= high_volume_threshold
prioritized = is_high_volume & has_severe_negative_gap

scoring_features["score"] = np.where(
    prioritized,
    scoring_features["log1p_impressions_mar"].astype(float),
    0.0,
)
scoring_features["reason_code"] = np.where(
    prioritized, "High-Visibility_Underperforming-CTR", ""
)
scoring_features["action_label"] = np.where(
    prioritized, "Review for Refresh", ""
)

queue = scoring_features.loc[prioritized].copy()
if len(queue) < 10:
    raise RuntimeError(
        f"The rule selected only {len(queue)} items; at least 10 are required for review."
    )
queue = queue.sort_values(
    ["score", "ctr_gap_pp", "content_hash_id"],
    ascending=[False, True, True],
    kind="mergesort",
).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
queue = queue[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action_label",
        "log1p_impressions_mar",
        "ctr_gap_pp",
    ]
]

repo_root = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "skills" / "README.md").is_file()
    ),
    Path.cwd(),
)
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

assert queue["score"].map(np.isfinite).all()
assert queue["reason_code"].eq("High-Visibility_Underperforming-CTR").all()
assert queue["action_label"].eq("Review for Refresh").all()
assert not {"target_decline", "april_impressions", "baseline_impressions"}.intersection(
    queue.columns
)
print(f"Selected {len(queue):,} qualifying items; exported ranked queue to {output_path}.")
print(
    f"March-only cutoffs: severe gap <= {SEVERE_CTR_GAP_THRESHOLD_PP:.4f} pp; "
    f"high volume >= {high_volume_threshold:.4f} among severe-gap items."
)

Selected 1,731 qualifying items; exported ranked queue to /workspaces/ml-starter-pack-flyrank-ai/work/outputs/baseline_action_score.csv.
March-only cutoffs: severe gap <= -0.0749 pp; high volume >= 5.1284 among severe-gap items.


## 3. Top 10 Review

The table is the first ten rows of the exported queue. The Markdown review list states the rule-based reason and one realistic failure mode for each pick. These are review prompts, not evidence that a refresh will cause recovery.

In [3]:
top_10 = queue.head(10).copy()
display(top_10)

possible_failure_modes = [
    "March impressions may reflect a short-lived campaign or seasonal spike rather than durable demand.",
    "The position-band CTR median may not represent this item's device, geography, or search-intent mix.",
    "A weak title or description could explain the CTR gap; rewriting the page body may not address it.",
    "SERP features or a changed branded-query mix may suppress clicks independently of content quality.",
    "An indexing or canonicalization problem may be the root cause, so a content refresh alone would miss it.",
    "A recent change after March may already have corrected the issue, but this rule intentionally uses March only.",
    "The item's aggregate impressions may be concentrated in one query or variant, hiding the true opportunity.",
    "Search intent may have shifted, making a different page format more appropriate than refreshing this one.",
    "A measurement change or incomplete daily observations may distort the March volume or CTR estimate.",
    "The page may already answer its main need; a low CTR gap could be caused by its search result presentation.",
]

review_lines = []
for row, failure_mode in zip(top_10.itertuples(index=False), possible_failure_modes):
    march_impressions = int(round(np.expm1(row.log1p_impressions_mar)))
    why = (
        f"March volume was about {march_impressions:,} impressions and the CTR gap "
        f"was {row.ctr_gap_pp:.2f} pp versus its position-band median."
    )
    review_lines.append(
        f"- Rank [{row.rank}]: Action: {row.action_label}. Why: {why} "
        f"What would make it wrong: {failure_mode}"
    )

display(Markdown("\n".join(review_lines)))

,rank,content_hash_id,score,reason_code,action_label,log1p_impressions_mar,ctr_gap_pp
0,1,content_bf078007df823490,10.707908,High-Visibility_Underperforming-CTR,Review for Refresh,10.707908,-0.07485
1,2,content_fa17add7836d36c3,9.440579,High-Visibility_Underperforming-CTR,Review for Refresh,9.440579,-0.07485
2,3,content_b154f6c2652cfeb9,9.336532,High-Visibility_Underperforming-CTR,Review for Refresh,9.336532,-0.07485
3,4,content_f5a7a2559d483a54,9.295325,High-Visibility_Underperforming-CTR,Review for Refresh,9.295325,-0.07485
4,5,content_520e203a08cd69ee,9.255601,High-Visibility_Underperforming-CTR,Review for Refresh,9.255601,-0.07485
5,6,content_d397987113cb84a0,9.199077,High-Visibility_Underperforming-CTR,Review for Refresh,9.199077,-0.07485
6,7,content_a27b382f00aa75c6,8.953769,High-Visibility_Underperforming-CTR,Review for Refresh,8.953769,-0.07485
7,8,content_1fdd52456a5d3f20,8.899731,High-Visibility_Underperforming-CTR,Review for Refresh,8.899731,-0.07485
8,9,content_83167156f76e33e5,8.828787,High-Visibility_Underperforming-CTR,Review for Refresh,8.828787,-0.07485
9,10,content_1bc8782404e3b132,8.664406,High-Visibility_Underperforming-CTR,Review for Refresh,8.664406,-0.07485


- Rank [1]: Action: Review for Refresh. Why: March volume was about 44,707 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: March impressions may reflect a short-lived campaign or seasonal spike rather than durable demand.
- Rank [2]: Action: Review for Refresh. Why: March volume was about 12,588 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: The position-band CTR median may not represent this item's device, geography, or search-intent mix.
- Rank [3]: Action: Review for Refresh. Why: March volume was about 11,344 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: A weak title or description could explain the CTR gap; rewriting the page body may not address it.
- Rank [4]: Action: Review for Refresh. Why: March volume was about 10,886 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: SERP features or a changed branded-query mix may suppress clicks independently of content quality.
- Rank [5]: Action: Review for Refresh. Why: March volume was about 10,462 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: An indexing or canonicalization problem may be the root cause, so a content refresh alone would miss it.
- Rank [6]: Action: Review for Refresh. Why: March volume was about 9,887 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: A recent change after March may already have corrected the issue, but this rule intentionally uses March only.
- Rank [7]: Action: Review for Refresh. Why: March volume was about 7,736 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: The item's aggregate impressions may be concentrated in one query or variant, hiding the true opportunity.
- Rank [8]: Action: Review for Refresh. Why: March volume was about 7,329 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: Search intent may have shifted, making a different page format more appropriate than refreshing this one.
- Rank [9]: Action: Review for Refresh. Why: March volume was about 6,827 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: A measurement change or incomplete daily observations may distort the March volume or CTR estimate.
- Rank [10]: Action: Review for Refresh. Why: March volume was about 5,792 impressions and the CTR gap was -0.07 pp versus its position-band median. What would make it wrong: The page may already answer its main need; a low CTR gap could be caused by its search result presentation.